In [5]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
import pandas as pd

In [2]:
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

In [4]:
pc.list_indexes()

IndexList([<name='customer-support', dim=512, ready=True>])

In [6]:
index = pc.Index("customer-support")

In [7]:
df = pd.read_csv("../data/unified_faq_dataset.csv")


def build_embedding_text(row, include_category=True):
    prefix = f"[{row['category']}]" if include_category else ""
    return f"{prefix} Q: {row['question']} A: {row['answer']}"


df["embedding_text"] = df.apply(build_embedding_text, axis=1)

In [11]:
metadata_list = df[["doc_id", "category", "difficulty", "source", "question", "answer"]].to_dict("records")

In [13]:
from tqdm import tqdm

BATCH_SIZE = 96
total = len(metadata_list)

for i in tqdm(range(0, total, BATCH_SIZE), desc="Upserting batches"):
    batch_metadata = metadata_list[i:i + BATCH_SIZE]
    records = [
        {
            "_id": str(metadata["doc_id"]),
            "text": df["embedding_text"].iloc[i + j],
            **metadata
        }
        for j, metadata in enumerate(batch_metadata)
    ]
    index.upsert_records(namespace="faq", records=records)

In [33]:
results = index.search(
    namespace="faq",
    top_k=2,
    inputs={"text": "How do I reset my password?"},
    fields=["text"]
)

# results.result["hits"]
for hit in results.result["hits"]:
    print(hit.id, hit.score, hit.fields["text"], sep=" | ", end="\n\n")

79 | 0.656317949295044 | [Account & Profile] Q: How can I reset my password? A: To reset your password, click on the 'Forgot Password' link on the login page and follow the instructions to reset your password.

34908 | 0.3595679998397827 | [Office Products] Q: Tried to set the combo and it's locked and I didn't set the password what should I do? A: Best suggestion I have found is at https://www.youtube.com/watch?v=FfeS_h3zIxQ Otherwise, you will have to try all 1000 combos and hope you get it quickly!

